# Suru voice bank generator (Ask SKY)

Clones the voice in your uploaded sample and speaks the Ask SKY chatbot's canned replies in that voice.

**What you do:**
1. Runtime > Run all (or Ctrl+F9)
2. When asked, upload the voice sample file (the .ogg voice note)
3. At the end `suru_voice_bank.zip` downloads automatically - send it back.

Uses a free GPU if Colab gives one; otherwise runs on CPU (slower, still works).

In [ ]:
#@title 1. Install packages (2-4 minutes)
# Safe to re-run: every step below is idempotent.
# Pin setuptools first - Colab's torch requires setuptools<82.
!pip install -q "setuptools<82" wheel jedi

# OpenVoice WITHOUT its pinned legacy deps (they break Colab's modern Python),
# then just the small set it actually needs.
!pip install -q --no-deps git+https://github.com/myshell-ai/OpenVoice.git
!pip install -q librosa pydub soundfile eng_to_ipa inflect unidecode pypinyin cn2an jieba langid wavmark

# MeloTTS with its normal dependencies
!pip install -q git+https://github.com/myshell-ai/MeloTTS.git
!python -m unidic download > /dev/null 2>&1

import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('cmudict', quiet=True)

# resemblyzer is only used for the optional A/B scoring in cell 7.
# If it can't build here, don't fail - cell 7 will just use the OpenVoice output.
import subprocess, sys
_r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'resemblyzer'], capture_output=True, text=True)
print('resemblyzer install:', 'ok' if _r.returncode == 0 else 'failed (fine - cell 7 will skip scoring)')
print('install done')

In [ ]:
#@title 2. Download OpenVoice V2 checkpoints
import os
os.makedirs('checkpoints_v2/converter', exist_ok=True)
os.makedirs('checkpoints_v2/base_speakers/ses', exist_ok=True)
base = 'https://huggingface.co/myshell-ai/OpenVoiceV2/resolve/main'
!wget -q "$base/converter/checkpoint.pth" -O checkpoints_v2/converter/checkpoint.pth
!wget -q "$base/converter/config.json" -O checkpoints_v2/converter/config.json
!wget -q "$base/base_speakers/ses/en-india.pth" -O checkpoints_v2/base_speakers/ses/en-india.pth
print('checkpoints done')

In [ ]:
#@title 3. Upload the voice sample
from google.colab import files
import subprocess
up = files.upload()  # pick the voice note file (e.g. audio-0ea39279.ogg)
src = list(up.keys())[0]
subprocess.run(['ffmpeg','-y','-i',src,'-ar','22050','-ac','1','suru_ref.wav'], check=True)
print('reference saved as suru_ref.wav')

In [ ]:
#@title 4. The reply bank (lines the chatbot speaks)
contact = ("The site lists these contact options. Email: ashwatthama seven one zero at gmail dot com. "
           "LinkedIn: linkedin dot com slash in slash akash dash sharma one nine nine eight. "
           "GitHub: github dot com slash akash eight eight three five.")

BANK = {
 "greeting": "Hi, I'm SKY. I can answer questions about Akash's experience, skills, education and product work. I only use verified details from this portfolio.",
 "hello": "Hello!",
 "zikpe": "At ZikPe, Akash contributes across the full product cycle: strategy, requirements, design, development and go-to-market. He owns PRDs and feature specs, helps prototype interfaces, drives KYC onboarding strategy with vendors, shapes corporate card work including tax-benefit and fuel cards, and contributes to product marketing as part of the core team.",
 "summary": "Akash has two plus years of product experience across fintech, AI and edtech. His portfolio lists Product Manager at ZikPe at ZikZuk, AI Product Manager Intern at Mactores, Product Analyst Intern at Labmentix, and Product Development and Strategy at Think And Learn, BYJU'S.",
 "byjus": "At Think And Learn, BYJU'S, Akash worked in Product Development and Strategy from January 2020 to March 2022. He contributed features for the tablet learning product, supported the Disney BYJU'S early-learning collaboration for K to 3 students, and worked with sales and IT teams to bring customer feedback into product decisions.",
 "labmentix": "At Labmentix, Akash was a Product Analyst Intern from May to August 2025. He gathered product requirements, developed web features with Python, Node.js, HTML, CSS and REST APIs, and used A/B testing and user-interaction data to assess improvements.",
 "mactores": "At Mactores, Akash was an AI Product Manager Intern from September 2025 to March 2026. He translated business requirements into analytics insights with product and data science teams, integrated AI and ML APIs into analytics workflows, and participated in agile planning and reviews.",
 "skills": "His listed strengths cover product strategy, PRDs and specs, roadmaps, research, prioritization, Agile and Scrum and go-to-market; UPI, KYC, corporate cards and fraud detection; generative AI, LLM and RAG fundamentals and prompt engineering; Python, SQL, JavaScript, Node.js, HTML, CSS and REST APIs; plus analytics tools including Power BI, Google Analytics and A/B testing.",
 "education": "Akash earned an MCA from New Horizon College of Engineering, VTU, Bengaluru, from 2023 to 2025, and a BCA from Acharya Institute of Technology, BU, Bengaluru, from 2016 to 2019.",
 "research": "Akash is a co-author of an IEEE conference paper on fraud detection in payment systems, published at ICEARS in February 2025.",
 "relocation": "Akash is based in Bengaluru, India, and the portfolio says he is open to relocation.",
 "joining": "Akash is an immediate joiner and has no notice period.",
 "resume": "You can use the Resume section to open or download Akash's one-page PDF. It covers ZikPe, Mactores, Labmentix, BYJU'S, technical skills, education and IEEE published research.",
 "contact": contact,
 "phone": "A phone number is not provided in the portfolio interface. " + contact,
 "fallback": "I do not want to guess about that. You can ask Akash directly. " + contact,
 "ai_error": "I'm having trouble reaching the AI right now. You can still ask about Akash's roles, ZikPe, BYJU'S, skills, education, research or contact details.",
}

print(len(BANK), 'lines')

In [ ]:
#@title 5. Generate with OpenVoice V2 (MeloTTS EN_INDIA base + tone-color conversion)
import os, torch
from melo.api import TTS
from openvoice.api import ToneColorConverter

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

converter = ToneColorConverter('checkpoints_v2/converter/config.json', device=device)
converter.load_ckpt('checkpoints_v2/converter/checkpoint.pth')
tgt_se = converter.extract_se(['suru_ref.wav'], se_save_path='suru_se.pth')
src_se = torch.load('checkpoints_v2/base_speakers/ses/en-india.pth', map_location=device)

model = TTS(language='EN', device=device)
spk_id = model.hps.data.spk2id['EN_INDIA']

os.makedirs('out_openvoice', exist_ok=True)
for key, text in BANK.items():
    tmp = f'_base_{key}.wav'
    model.tts_to_file(text, spk_id, tmp, speed=0.95, quiet=True)
    converter.convert(tmp, src_se, tgt_se, f'out_openvoice/{key}.wav', tau=0.3, message='suru')
    os.remove(tmp)
    print('done', key)
print('OpenVoice bank complete')

In [ ]:
#@title 6. Optional A/B: Chatterbox Multilingual (set RUN_CHATTERBOX = False to skip)
RUN_CHATTERBOX = True  #@param {type:"boolean"}

if RUN_CHATTERBOX:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chatterbox-tts'], check=False)
    try:
        import torch, os, torchaudio
        from chatterbox.mtl_tts import ChatterboxMultilingualTTS
        cb = ChatterboxMultilingualTTS.from_pretrained(device='cuda' if torch.cuda.is_available() else 'cpu')
        os.makedirs('out_chatterbox', exist_ok=True)
        for key, text in BANK.items():
            wav = cb.generate(text, language_id='en', audio_prompt_path='suru_ref.wav')
            torchaudio.save(f'out_chatterbox/{key}.wav', wav, cb.sr)
            print('done', key)
        print('Chatterbox bank complete')
    except Exception as e:
        print('Chatterbox unavailable - continuing with OpenVoice only. Error:', e)

In [ ]:
#@title 7. Score A vs B against the reference, pick the better clone
import os, glob, json, shutil

scores = {}
try:
    from resemblyzer import VoiceEncoder, preprocess_wav
    import numpy as np
    enc = VoiceEncoder()
    ref = enc.embed_utterance(preprocess_wav('suru_ref.wav'))

    def score(folder):
        sims = {}
        for f in sorted(glob.glob(f'{folder}/*.wav')):
            try:
                e = enc.embed_utterance(preprocess_wav(f))
                sims[os.path.basename(f)[:-4]] = float(np.inner(ref, e))
            except Exception:
                pass
        return sims

    scores['openvoice'] = score('out_openvoice')
    if glob.glob('out_chatterbox/*.wav'):
        scores['chatterbox'] = score('out_chatterbox')
    for name, s in scores.items():
        if s:
            print(name, 'mean similarity:', round(sum(s.values())/len(s), 3))
except Exception as e:
    print('scoring skipped:', e)

scores = {k: v for k, v in scores.items() if v}
winner = max(scores, key=lambda n: sum(scores[n].values())/len(scores[n])) if scores else 'openvoice'
print('WINNER:', winner)

os.makedirs('final', exist_ok=True)
manifest = {}
order = ('chatterbox','openvoice') if winner=='chatterbox' else ('openvoice','chatterbox')
for key in BANK:
    for name in order:
        src = f'out_{name}/{key}.wav'
        if os.path.exists(src):
            shutil.copy(src, f'final/{key}.wav')
            manifest[key] = {'engine': name, 'similarity': scores.get(name,{}).get(key)}
            break
json.dump({'winner': winner, 'files': manifest}, open('final/manifest.json','w'), indent=2)
print('final/ ready')

In [ ]:
#@title 8. Convert to mp3, zip, download
import glob, subprocess, os
for f in glob.glob('final/*.wav'):
    out = f[:-4] + '.mp3'
    subprocess.run(['ffmpeg','-y','-i',f,'-codec:a','libmp3lame','-b:a','96k',out], check=True, capture_output=True)
    os.remove(f)
!cd final && zip -q ../suru_voice_bank.zip *.mp3 manifest.json
from google.colab import files
files.download('suru_voice_bank.zip')
print('done - send suru_voice_bank.zip back')